<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Synthetic Financial Time Series

**Sequential GAN: LSTM-Based Generation of Return Paths (PyTorch)**

&copy; Dr. Yves J. Hilpisch

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>

## Imports & Settings

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pylab import plt, mpl

In [ ]:
import torch

In [ ]:
sys.path.insert(0, os.path.abspath('..'))
from synthts.data import load_eod, compute_log_returns, ReturnScaler
from synthts.sequential import (SeqGenerator, SeqDiscriminator,
    train_seq_gan, generate_sequences)
from synthts.evaluate import (ks_test, ks_test_batch, pass_rate,
    describe_comparison, plot_histograms, plot_sorted,
    plot_pvalue_histogram,
    compare_acf, plot_acf_comparison, plot_drawdown_comparison)

In [ ]:
plt.style.use('seaborn-v0_8')
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['font.family'] = 'serif'

## Data Preparation

In [ ]:
prices = load_eod(symbol='GLD', n_obs=2 * 252)
rets = compute_log_returns(prices)
scaler = ReturnScaler()
rets_ = scaler.fit_transform(rets)

In [ ]:
f'Returns: {len(rets)} observations'

## Model Training

In [ ]:
SEQ_LEN = 63  #  # 3-month window
LATENT_DIM = 4
HU = 64

In [ ]:
g = SeqGenerator(latent_dim=LATENT_DIM, hidden_units=HU,
                 output_dim=1, seq_len=SEQ_LEN, num_layers=2)
d = SeqDiscriminator(input_dim=1, hidden_units=HU, num_layers=2)

In [ ]:
%%time
history = train_seq_gan(
    g, d, rets_.reshape(-1, 1),
    seq_len=SEQ_LEN, epochs=5001, batch_size=32,
    lr_d=0.001, lr_g=0.0001, latent_dim=LATENT_DIM,
    seed=100
)

## Generating Synthetic Sequences

In [ ]:
N = 25
seqs = generate_sequences(g, N, latent_dim=LATENT_DIM)
seqs.shape

In [ ]:
# flatten sequences for marginal comparison
synth_flat = scaler.inverse_transform(seqs[:, :, 0].flatten())
data = pd.DataFrame({'real': rets})
for i in range(min(N, 10)):
    data[f'synth_{i:02d}'] = scaler.inverse_transform(
        seqs[i, :, 0])

### Marginal Distribution

In [ ]:
plot_histograms(rets, synth_flat);

In [ ]:
plot_sorted(rets, synth_flat);

### Temporal Structure — ACF Comparison

In [ ]:
comp = compare_acf(rets, synth_flat, max_lag=20)
f"ACF diff: {comp['acf_diff']:.4f}, " \
f"Squared-return ACF diff: {comp['acf2_diff']:.4f}"

In [ ]:
plot_acf_comparison(rets, synth_flat, max_lag=20);

### Drawdown Comparison

In [ ]:
plot_drawdown_comparison(rets, synth_flat, n_bootstrap=200);

### KS Test

In [ ]:
synth_cols = [c for c in data.columns if c.startswith('synth')]
pvals = ks_test_batch(data['real'].values, data[synth_cols])
f'KS pass rate: {pass_rate(pvals):.0%}'

In [ ]:
plot_pvalue_histogram(pvals);

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>